# Instructor Solution: Day 3 Selenium Student Exercise

This is the instructor answer key for `day_3_selenium_student_exercise.ipynb`.

It solves the browser workflow against the same local practice page used in the student exercise. Keep this separate from the student notebook.

The key answers are:

- keyword input: `#query-box`
- submit button: `#run-filter`
- create-result button: `#create-result`
- message text: `#message`
- repeated result cards: `.result-card`
- title link inside each card: `a.item-link`
- evidence checkpoint: `#evidence-end`
- hidden attributes: `data-record-id`, `data-keyword`


In [ ]:
from pathlib import Path
from urllib.parse import quote
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

outdir = Path("data")
raw_dir = outdir / "raw"
processed_dir = outdir / "processed"
raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


## Same Local Practice Page

The solution uses the same local HTML page as the student exercise. No live website is contacted.


In [ ]:
exercise_html = """
<!doctype html>
<html>
<head>
  <title>Research Task Selenium Exercise</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 32px; line-height: 1.4; }
    .control-panel { border: 2px solid #555; padding: 16px; margin-bottom: 20px; max-width: 820px; }
    input { font-size: 18px; padding: 7px; width: 360px; }
    button { font-size: 17px; padding: 8px 12px; margin: 6px; cursor: pointer; }
    #message { background: #eef; padding: 10px; font-weight: bold; }
    .result-card { border: 1px solid #888; padding: 12px; margin: 10px 0; max-width: 650px; background: #fafafa; }
    .result-card a { font-weight: bold; }
    .scroll-zone { height: 850px; background: linear-gradient(#fff, #eee); padding-top: 30px; color: #555; }
    #evidence-end { border-top: 4px solid #444; padding-top: 16px; }
  </style>
</head>
<body>
  <h1>Research Task Exercise Page</h1>
  <p>This local page behaves like a tiny interactive research-task tracker.</p>

  <div class="control-panel">
    <label for="query-box">Search keyword</label><br>
    <input id="query-box" name="keyword" placeholder="enter a keyword">
    <button id="run-filter">Run filter</button>
    <button id="create-result">Create result</button>
    <p id="message">No keyword submitted yet.</p>
  </div>

  <section id="result-list" aria-label="research results">
    <article class="result-card" data-record-id="result-1" data-keyword="governance">
      <a class="item-link" href="/records/1">Governance reading note</a>
      <p>Initial visible result.</p>
    </article>
  </section>

  <div class="scroll-zone">Scroll down after creating results.</div>

  <h2 id="evidence-end">Evidence Checkpoint</h2>
  <p>This is where you should scroll before taking a screenshot.</p>

  <script>
    let resultCounter = 1;

    document.querySelector('#run-filter').addEventListener('click', function() {
      const keyword = document.querySelector('#query-box').value;
      document.querySelector('#message').textContent = 'Active keyword: ' + keyword;
    });

    document.querySelector('#query-box').addEventListener('keydown', function(event) {
      if (event.key === 'Enter') {
        document.querySelector('#run-filter').click();
      }
    });

    document.querySelector('#create-result').addEventListener('click', function() {
      resultCounter = resultCounter + 1;
      const keyword = document.querySelector('#query-box').value || 'unlabeled';
      const card = document.createElement('article');
      card.className = 'result-card';
      card.setAttribute('data-record-id', 'result-' + resultCounter);
      card.setAttribute('data-keyword', keyword);
      card.innerHTML = '<a class="item-link" href="/records/' + resultCounter + '">Generated result ' + resultCounter + '</a>' +
                       '<p>Generated for keyword: ' + keyword + '</p>';
      document.querySelector('#result-list').appendChild(card);
    });
  </script>
</body>
</html>
"""

exercise_url = "data:text/html;charset=utf-8," + quote(exercise_html)
print(exercise_url[:120] + "...")


## Compact Selector Answer Key

This cell records the selector and attribute choices separately, so you can quickly compare them with student attempts.


In [ ]:
keyword = "platform accountability"
number_of_clicks = 3

selectors = {
    "keyword_input": "#query-box",
    "run_filter_button": "#run-filter",
    "create_result_button": "#create-result",
    "message": "#message",
    "result_card": ".result-card",
    "result_title": "a.item-link",
    "evidence_checkpoint": "#evidence-end",
}

record_id_attribute = "data-record-id"
keyword_attribute = "data-keyword"

selectors


## Full Worked Solution

This cell completes the same workflow students are asked to build: open, type, submit, wait, click, extract, scroll, screenshot, save CSV, and close the browser.


In [ ]:
options = Options()
options.add_argument("--window-size=1100,850")

# Use a visible browser for teaching. Add "--headless=new" if you want it to run quietly.
# options.add_argument("--headless=new")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 10)

try:
    # Open the local page in the browser.
    driver.get(exercise_url)

    # Find the keyword input, click into it, and type the keyword.
    keyword_input = wait.until(
        EC.presence_of_element_located((By.CSS_SELECTOR, selectors["keyword_input"]))
    )
    keyword_input.click()
    keyword_input.send_keys(keyword)

    # Submit the form by pressing Enter. Clicking #run-filter would also work.
    keyword_input.send_keys(Keys.ENTER)

    # Wait for evidence that the page reacted to the input.
    wait.until(
        EC.text_to_be_present_in_element((By.CSS_SELECTOR, selectors["message"]), keyword)
    )
    message_text = driver.find_element(By.CSS_SELECTOR, selectors["message"]).text
    print(message_text)

    # Click the Create result button several times to generate new DOM elements.
    create_button = driver.find_element(By.CSS_SELECTOR, selectors["create_result_button"])
    for click_number in range(number_of_clicks):
        create_button.click()
        time.sleep(0.5)

    # Extract visible link text plus hidden data attributes from each result card.
    result_rows = []
    result_cards = driver.find_elements(By.CSS_SELECTOR, selectors["result_card"])
    for card in result_cards:
        title_link = card.find_element(By.CSS_SELECTOR, selectors["result_title"])
        result_rows.append(
            {
                "record_id": card.get_attribute(record_id_attribute),
                "keyword": card.get_attribute(keyword_attribute),
                "result_title": title_link.text,
                "link_href": title_link.get_attribute("href"),
            }
        )

    results_df = pd.DataFrame(result_rows)
    display(results_df)

    # Scroll to the evidence checkpoint before saving the screenshot.
    evidence_checkpoint = driver.find_element(By.CSS_SELECTOR, selectors["evidence_checkpoint"])
    driver.execute_script("arguments[0].scrollIntoView();", evidence_checkpoint)
    time.sleep(1)

    # Save visual and tabular evidence.
    screenshot_path = raw_dir / "notebook_selenium_student_exercise_solution.png"
    results_path = processed_dir / "notebook_selenium_student_results_solution.csv"

    driver.get_screenshot_as_file(str(screenshot_path))
    results_df.to_csv(results_path, index=False)

    print("Saved screenshot:", screenshot_path)
    print("Saved result table:", results_path)

finally:
    driver.quit()


## Alternative Submission Route

If a student clicks the Run filter button instead of pressing Enter, the submission part can be written this way:


In [ ]:
# Alternative to keyword_input.send_keys(Keys.ENTER):
# run_filter_button = driver.find_element(By.CSS_SELECTOR, selectors["run_filter_button"])
# run_filter_button.click()
